In [ ]:
!pip install torch scikit-learn

In [ ]:
# --- Tools we need for our exercise---
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1) LOAD THE DATA
data     = fetch_california_housing()
features = data.data  # 8 input numbers for each house
prices   = data.target.reshape(-1, 1)  # Reshape per lo scaler

print(f"Number of samples: {len(prices)}")
print(f"Number of features: {features.shape[1]}")
print(f"Feature names: {data.feature_names}")

In [ ]:
# 2) SPLIT THE DATA (aggiungiamo anche validation set)
# We split our data into training, validation and test sets.
train_features, temp_features, train_prices, temp_prices = train_test_split(
    features, prices, test_size=0.3, random_state=42
)
val_features, test_features, val_prices, test_prices = train_test_split(
    temp_features, temp_prices, test_size=0.5, random_state=42
)

print(f"Training samples: {len(train_prices)}")
print(f"Validation samples: {len(val_prices)}")
print(f"Test samples: {len(test_prices)}")

In [ ]:
# We rescale the data so all numbers are comparable
# (mean = 0, standard deviation = 1)

feature_scaler = StandardScaler()
price_scaler = StandardScaler()

# Scale input data (house features)
train_features = feature_scaler.fit_transform(train_features)
val_features   = feature_scaler.transform(val_features)
test_features  = feature_scaler.transform(test_features)

# Scale output data (house prices)
train_prices = price_scaler.fit_transform(train_prices)
val_prices   = price_scaler.transform(val_prices)
test_prices  = price_scaler.transform(test_prices)

print("Data scaled: all values now have similar size")


In [ ]:
# 4) CONVERT TO PYTORCH TENSORS
train_features = torch.tensor(train_features, dtype=torch.float32)
train_prices   = torch.tensor(train_prices, dtype=torch.float32)

val_features = torch.tensor(val_features, dtype=torch.float32)
val_prices = torch.tensor(val_prices, dtype=torch.float32)

test_features  = torch.tensor(test_features, dtype=torch.float32)
test_prices_scaled = torch.tensor(test_prices_scaled, dtype=torch.float32)

print(f"Train features shape: {train_features.shape}")
print(f"Train prices shape: {train_prices.shape}")
print(f"Val features shape: {val_features.shape}")

In [ ]:
# 5) MODELLO PIÙ SEMPLICE E CON DROPOUT
class HousePriceModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        
        # Modello semplificato con dropout per regolarizzazione
        self.network = nn.Sequential(
            nn.Linear(n_features, 128), #First layer from input to 128 neurons
            nn.ReLU(),                  #Non-linear activation

            nn.Linear(128, 64),         #Second layer from 128 to 64 neurons
            nn.ReLU(),                  #Non-linear activation

            nn.Linear(64, 32),          #Third layer from 64 to 32 neurons
            nn.ReLU(),                  #Non-linear activation

            nn.Linear(32, 32),          #Fourth layer from 32 to 32 neurons
            nn.ReLU(),                  #Non-linear activation

            nn.Linear(32, 1)            #Output layer to single value
        )

    def forward(self, x):
        return self.network(x)


model = HousePriceModel(train_features.shape[1])
print(model)

In [ ]:
# We measure how wrong the predictions are
loss_function = nn.MSELoss()

# How big each learning step is
learning_rate = 0.01

# Tool that updates the model to make predictions better
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# How many times we repeat the learning process
number_of_epochs = 10000

for epoch in range(number_of_epochs):

    # --- TRAINING ---
    model.train()                         # learning mode
    predicted_prices = model(train_features)
    train_loss = loss_function(predicted_prices, train_prices)

    optimizer.zero_grad()                 # reset old gradients
    train_loss.backward()                 # understand mistakes
    optimizer.step()                      # improve the model

    # --- VALIDATION ---
    model.eval()                          # evaluation mode
    with torch.no_grad():
        val_predictions = model(val_features)
        val_loss = loss_function(val_predictions, val_prices)

    # Print progress sometimes
    if epoch % 50 == 0:
        print(
            f"Epoch {epoch} | "
            f"Train error: {train_loss.item():.4f} | "
            f"Validation error: {val_loss.item():.4f}"
        )


In [ ]:
# 7) EVALUATE 

# At first, we switch the model to evaluation mode.
model.eval()
with torch.no_grad():
    test_predictions_scaled = model(test_features)
    
    # De-scale the predictions
    test_predictions = target_scaler.inverse_transform(test_predictions_scaled.numpy())
    
    # Metrics on original values
    test_prices_original = test_prices  # Already in original scale
    mae = torch.mean(torch.abs(torch.tensor(test_predictions) - torch.tensor(test_prices_original)))
    rmse = torch.sqrt(torch.mean((torch.tensor(test_predictions) - torch.tensor(test_prices_original)) ** 2))
    mape = torch.mean(torch.abs((torch.tensor(test_predictions) - torch.tensor(test_prices_original)) / torch.tensor(test_prices_original))) * 100

print(f"\n=== Final Results ===")
print(f"MAE:  ${mae.item() * 100000:,.0f}")
print(f"RMSE: ${rmse.item() * 100000:,.0f}")
print(f"MAPE: {mape.item():.2f}%")